In [1]:
from dotenv import load_dotenv
load_dotenv("/mnt/disk1/project/project/aikb/.env")


True

In [2]:
import polars as pl
from datalake import scan_table, get_catalog

catalog = get_catalog()

In [3]:

rename_map = {
    "participant.p131286": "htn_date",
    "participant.p31": "sex",
    "participant.p21003_i0": "age",
    "participant.p21001_i0": "bmi",
    "participant.p1239_i0": "smoking_current",
    "participant.p1249_i0": "smoking_past",
    "participant.p20117_i0": "smoking_pack_years",
    "participant.p6138_i0": "education",
    "participant.p738_i0": "income",
    "participant.p22189": "townsend_di",
    "participant.p54_i0": "assessment_centre",
    "participant.p4080_i0_a0": "systolic_bp_1",
    "participant.p4079_i0_a0": "diastolic_bp_1",
    "participant.p4080_i0_a1": "systolic_bp_2",
    "participant.p4079_i0_a1": "diastolic_bp_2",
    "participant.p2966_i0": "age_htn_diagnosed",
}


hp_df = (
    scan_table("ukb.hypertension_cohort")
    .rename(rename_map)  # Rename field: ukb code to readable name
    .with_columns(  # Processing education field
        pl.col("education")
        .str.strip_chars("[]")
        .str.split(",")
        .list.eval(pl.element().cast(pl.Int32))
        .list.min()
        .replace(-7, 0)  # -7 (无学历) 映射为 0
    ).with_columns(
        pl.col('htn_date').is_not_null().alias('hpt')
    )
)

olink_df = scan_table("ukb.hypertension_cohort").with_columns(
    pl.col("eid").cast(pl.String)
)  # convert eid datatype



In [6]:
# catalog.drop_table("ukb.hpt_cov_clean")
join_tb = catalog.create_table_if_not_exists(
    "ukb.hpt_cov_clean", schema=hp_df.schema.to_arrow()
)
hp_df.collect().write_iceberg(join_tb, "append")

/tmp/ipykernel_441483/2330742085.py:3: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  "ukb.hpt_cov_clean", schema=hp_df.schema.to_arrow()
Unable to resolve region for bucket warehouse


## 蛋白质组Logistic回归

In [ ]:
from typing import List
from polars import LazyFrame
import statsmodels.api as sm

def logit_protein(df: LazyFrame, protein: str, outcome_field: str,covariance_field: List[str] ):
    fields = [protein, *covariance_field, outcome_field]
    clean_df = df.select(pl.col(fields)).drop_nans().drop_nulls()
    X = clean_df.select(pl.col([protein, *covariance_field]).cast(pl.Float64)).collect().to_numpy()
    X = sm.add_constant(X)
    y = clean_df.select([outcome_field]).collect().to_numpy()

    model = sm.Logit(y, X)
    result = model.fit(disp=False)
    return result
    

In [15]:
covariance_field = [
    "sex",
    "age",
    "bmi",
    "smoking_current",
    "smoking_past",
    "smoking_pack_years",
    "education",
    "income"
]
result = logit_protein(
    df=df, protein="a1bg", outcome_field="hpt", covariance_field=covariance_field
)

In [16]:
result.summary().tables[1]

,coef,std err,z,P>|z|,[0.025,0.975]
const,-7.7911,0.133,-58.734,0.000,-8.051,-7.531
x1,0.5658,0.062,9.092,0.000,0.444,0.688
x2,0.4865,0.024,20.675,0.000,0.440,0.533
x3,0.0764,0.002,50.823,0.000,0.073,0.079
x4,0.1205,0.003,47.910,0.000,0.116,0.125
x5,-0.0305,0.032,-0.949,0.342,-0.093,0.032
x6,-0.0517,0.009,-6.028,0.000,-0.069,-0.035
x7,-0.1241,0.023,-5.474,0.000,-0.169,-0.080
x8,0.0033,0.006,0.531,0.596,-0.009,0.015
x9,-0.0413,0.005,-7.605,0.000,-0.052,-0.031


In [30]:
schema_dic = olink_df.schema.to_frame().to_dict()

protein_fields = [key[0] for key in schema_dic.items() if key[0] != 'eid']

/tmp/ipykernel_3236523/1698842242.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.


In [ ]:
import gc
import numpy as np


def extract_result_to_df(result, protein: str) -> pl.DataFrame:
    params = result.params
    bse = result.bse
    conf_int = result.conf_int()
    pvalues = result.pvalues

    var_names = ["const", protein, *covariance_field]

    rows = []
    for idx, var in enumerate(var_names):
        rows.append({
            "protein": protein,
            "variable": var,
            "coef": float(params[idx]),
            "std_err": float(bse[idx]),
            "z": float(params[idx] / bse[idx]),
            "pvalue": float(pvalues[idx]),
            "ci_lower": float(conf_int[idx][0]),
            "ci_upper": float(conf_int[idx][1]),
        })
    return pl.DataFrame(rows)


In [4]:
catalog.drop_table("ukb.pwas_hypertension_logit")


In [ ]:

for index, i in enumerate(protein_fields):
    result = logit_protein(
        df=df, protein=i, outcome_field="hpt", covariance_field=covariance_field
    )
    result_df = extract_result_to_df(result, i)
    pwas_table = catalog.create_table_if_not_exists("ukb.pwas_hypertension_logit", schema=result_df.schema.to_arrow())
    result_df.write_iceberg(pwas_table, 'append')
    del result, result_df
    gc.collect()
    print(f"Processing: {index}/{len(protein_fields)}")



Processing: 0/2923
Processing: 1/2923
Processing: 2/2923
Processing: 3/2923
Processing: 4/2923
Processing: 5/2923
Processing: 6/2923
Processing: 7/2923
Processing: 8/2923
Processing: 9/2923
Processing: 10/2923
Processing: 11/2923
Processing: 12/2923
Processing: 13/2923
Processing: 14/2923
Processing: 15/2923
Processing: 16/2923
Processing: 17/2923
Processing: 18/2923
Processing: 19/2923
Processing: 20/2923
Processing: 21/2923
Processing: 22/2923
Processing: 23/2923
Processing: 24/2923
Processing: 25/2923
Processing: 26/2923
Processing: 27/2923
Processing: 28/2923
Processing: 29/2923
Processing: 30/2923
Processing: 31/2923
Processing: 32/2923
Processing: 33/2923
Processing: 34/2923
Processing: 35/2923
Processing: 36/2923
Processing: 37/2923
Processing: 38/2923
Processing: 39/2923
Processing: 40/2923
Processing: 41/2923
Processing: 42/2923
Processing: 43/2923
Processing: 44/2923
Processing: 45/2923
Processing: 46/2923
Processing: 47/2923
Processing: 48/2923
Processing: 49/2923
Processing

In [35]:
catalog.list_tables('ukb')
tb = catalog.load_table("ukb.pwas_hypertension_logit")
pwas_df = pl.scan_iceberg(tb)

In [36]:
pwas_df.head().collect()

protein,variable,coef,std_err,z,pvalue,ci_lower,ci_upper
str,str,f64,f64,f64,f64,f64,f64
"""gli2""","""const""",-7.894111,0.134231,-58.809765,0.0,-8.1572,-7.631022
"""gli2""","""gli2""",-0.04458,0.032284,-1.380844,0.167327,-0.107855,0.018696
"""gli2""","""sex""",0.422075,0.022752,18.551511,7.9292e-77,0.377483,0.466667
"""gli2""","""age""",0.077878,0.001521,51.203573,0.0,0.074897,0.080859
"""gli2""","""bmi""",0.123147,0.002536,48.566735,0.0,0.118178,0.128117
